In [ ]:
import os
import torch
from torch import nn, optim
import helper_utils
from model import MyModel
from dataset import QuickDrawDataset, SubsetQuickDraw
import importlib
importlib.reload(helper_utils)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Prepare Dataset To Train

In [ ]:
root_dir = "dataset"
dataset = QuickDrawDataset(root_dir)

In [ ]:
# mean, std = helper_utils.get_mean_std(dataset)
mean = torch.tensor([0.1577, 0.1577, 0.1577])
std = torch.tensor([0.3355, 0.3355, 0.3355])
# # tensor([0.1577, 0.1577, 0.1577])
# # tensor([0.3355, 0.3355, 0.3355])

In [ ]:
train_transform, val_transform = helper_utils.define_transform(mean, std)
train_dataset, val_dataset, test_dataset = helper_utils.create_data_splits(dataset, 0.1, 0.1)
train_dataset = SubsetQuickDraw(train_dataset, train_transform)
val_dataset = SubsetQuickDraw(val_dataset, val_transform)
test_dataset = SubsetQuickDraw(test_dataset, val_transform)

In [ ]:
batch_size = 32
train_loader, val_loader, test_loader = helper_utils.get_data_loader(train_dataset, val_dataset, test_dataset, batch_size)
print(f"Lenght of train dataset:            {len(train_dataset)}")
print(f"Length of val dataset:              {len(val_dataset)}")
print(f"Length of test dataset:             {len(test_dataset)}")

## Training

In [ ]:
num_classes = len(dataset.classes)
dropout = 0.3
EPOCHS = 30
save_path = "checkpoints"
model = MyModel(num_classes=num_classes, dropout=dropout)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max")
loss_fn = nn.CrossEntropyLoss()
history = helper_utils.train(model=model, optimizer=optimizer, loss_fn=loss_fn,
                                    train_loader=train_loader, val_loader=val_loader,
                                    epochs=EPOCHS, device=device, scheduler=scheduler, save_path=save_path)

In [ ]:
helper_utils.plot_history(history)

In [ ]:
num_classes = 25
model = MyModel(num_classes)
state_dict = torch.load(r"checkpoints\best.pth")
model.load_state_dict(state_dict)

In [ ]:
img_path = r"test\test8.png"
score, label = helper_utils.inference(img_path, model, device)
print(f"Label {label}, score {score}")